# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook shows how to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing entities by their `@id` fields for reproducible and metadata-driven data science.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a DatasetMetadata object

# Print high-level dataset information
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'date_published', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their '@id' and fields

print("Available Record Sets:")
record_sets = list(dataset.record_sets.keys())  # list of @id for record sets
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- Record Set @id: {rs_id}")
    print(f"  Name: {getattr(record_set, 'name', '')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    # List fields by their @id
    print(f"  Fields (@id):")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - {field.id} (name: {field.name}, dataType: {field.data_type})")
    print()

# Show a sample record structure from each record set
for rs_id in record_sets:
    print(f"First record from record set {rs_id}:")
    try:
        record_iter = dataset.records(record_set=rs_id)
        sample = next(record_iter)
        pprint.pprint(sample)
    except Exception as e:
        print(f"  (Could not load sample: {e})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this demonstration, select the main protein data record set by @id as shown in the overview above.
# Usually, mass spectrometry datasets have a main record set with fields such as accession, description, peptide counts, MW, etc.
# For this dataset, let's auto-select the largest record set for demonstration:

dfs = {}
rows_count = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        dfs[rs_id] = pd.DataFrame(records)
        rows_count[rs_id] = len(dfs[rs_id])
        print(f"Loaded record set {rs_id} with {len(dfs[rs_id])} records and {dfs[rs_id].shape[1]} columns.")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Pick the record set with the most rows as the primary protein table (typical for MS datasets)
main_rs_id = max(rows_count, key=rows_count.get)

print(f"\nPrimary table selected for analysis: {main_rs_id}\nColumns available (by field @id):")
print(dfs[main_rs_id].columns.tolist())
dfs[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will reference all fields by their `@id` as described by the Croissant schema for this dataset.

In [ ]:
# Show numeric fields by their @id
numeric_id_candidates = []
for field in dataset.record_sets[main_rs_id].fields:
    if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
        numeric_id_candidates.append(field.id)
print("Numeric fields (@id):", numeric_id_candidates)

# Use the first numeric field for filtering/normalization demo
if numeric_id_candidates:
    numeric_field_id = numeric_id_candidates[0]
else:
    raise ValueError("No numeric fields found in this dataset.")
print(f"Using field '@id': {numeric_field_id}")

# Filter: keep rows with value > 10 for this numeric field
threshold = 10
filtered_df = dfs[main_rs_id][dfs[main_rs_id][numeric_field_id] > threshold]
print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
print(filtered_df.head())

# Normalize the selected field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (pick first non-numeric)
group_field = None
for field in dataset.record_sets[main_rs_id].fields:
    # string, text, etc
    if field.data_type not in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
        group_field = field.id
        break
if group_field and group_field in dfs[main_rs_id].columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by '{group_field}' (showing mean for {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], bins=40, kde=True)
plt.title(f"Distribution of values for {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
    plt.title(f"Distribution of {numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load the FAIR² mass spectrometry dataset with `mlcroissant`, explored record sets and their schema using `@id` for reference, conducted basic EDA, and visualized protein abundance features. Every field, record set, and transformation can be reproducibly referenced using their Croissant `@id`, facilitating FAIR and transparent data analysis.